# DeepSORT Final Project — Colab

**Runtime:** GPU (T4). **Данные:** `MyDrive/Videos-CV/`

Запускайте ячейки **по порядку**.

In [ ]:
# 1. GPU
!nvidia-smi

In [ ]:
# 2. Клонирование и установка
REPO_URL = "https://github.com/danvlak-batya/deep-sort-project.git"
PROJECT_DIR = "/content/deep-sort-project"

import os
os.chdir("/content")

if not os.path.exists(os.path.join(PROJECT_DIR, "run_tracker.py")):
    if os.path.exists(PROJECT_DIR):
        !rm -rf {PROJECT_DIR}
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print("Папка:", os.getcwd())

# torchreid с PyPI не ставится на Python 3.12 — используем install_colab_deps.py
!python tools/install_colab_deps.py

import ultralytics
import torchreid
print("Проверка: ultralytics и torchreid импортируются — OK")

In [ ]:
# 3. Google Drive
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Videos-CV"

import os
assert os.path.isdir(DATA_ROOT)
for name in sorted(os.listdir(DATA_ROOT)):
    p = os.path.join(DATA_ROOT, name)
    if os.path.isdir(p):
        print(name)

In [ ]:
# 4. Настройки
import os
from utils.mot_paths import find_sequence_dir, list_image_filenames

SEQUENCE_FOLDER = "Kitti-17"
SEQUENCE = "KITTI-17"
DETECTOR = "yolov8n"
REID = "osnet_x0_25"

SEQ_DIR = find_sequence_dir(DATA_ROOT, SEQUENCE_FOLDER)
print("Кадров:", len(list_image_filenames(SEQ_DIR)))

In [ ]:
# 5. Трекинг
import os
os.makedirs("results/demo", exist_ok=True)

!python run_tracker.py \
  --sequence_dir "{SEQ_DIR}" \
  --config configs/default.yaml \
  --detector {DETECTOR} \
  --reid {REID} \
  --output_file results/demo/{SEQUENCE}.txt

p = f"results/demo/{SEQUENCE}.txt"
print("OK:", os.path.exists(p), "size:", os.path.getsize(p) if os.path.exists(p) else 0)

In [ ]:
# 6. Overlay-видео
!python tools/generate_overlays.py \
  --mot_dir "{DATA_ROOT}" \
  --results_dir results/demo \
  --output_dir results/overlays \
  --sequence {SEQUENCE}

from IPython.display import Video, display
import os
v = f"results/overlays/{SEQUENCE}_overlay.mp4"
display(Video(v, embed=True)) if os.path.exists(v) else print("Нет видео — проверьте шаг 5")

In [ ]:
# 7. Все 6 видео + HOTA (долго)
RUN_FULL = False
if RUN_FULL:
    !python eval/run_benchmark.py --mot_root "{DATA_ROOT}" --output_dir results/modern
    !python eval/run_mot.py --mot_dir "{DATA_ROOT}" --output_dir results/modern --skip_tracking